In [ ]:
import pandas as pd
import numpy as np

from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

In [ ]:
df = pd.read_csv("phd_student_data.csv")
df.head()

In [ ]:
date_columns = [
    "Registration_Date",
    "Proposal_Review_Date",
    "Research_Review_Date",
    "Submission_Date",
    "Viva_Date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col])

In [ ]:
def get_semester_index(date):
    year = date.year
    month = date.month
    
    # Semester I: September (9) to February (2)
    if month >= 9:
        semester = 1
        academic_year = year
    elif month <= 2:
        semester = 1
        academic_year = year - 1
    else:
        # March to August
        semester = 2
        academic_year = year - 1
    
    return academic_year * 2 + semester

df["reg_sem"] = df["Registration_Date"].apply(get_semester_index)
df["proposal_sem"] = df["Proposal_Review_Date"].apply(get_semester_index)
df["research_sem"] = df["Research_Review_Date"].apply(get_semester_index)
df["submission_sem"] = df["Submission_Date"].apply(get_semester_index)
df["viva_sem"] = df["Viva_Date"].apply(get_semester_index)

df["proposal_delay_sem"] = df["proposal_sem"] - df["reg_sem"]
df["research_delay_sem"] = df["research_sem"] - df["proposal_sem"]
df["submission_delay_sem"] = df["submission_sem"] - df["research_sem"]
df["total_duration_sem"] = df["viva_sem"] - df["reg_sem"]

In [ ]:
def create_got_label(row):
    if row["Type"] == "Full Time":
        return 1 if row["total_duration_sem"] <= 10 else 0
    else:
        return 1 if row["total_duration_sem"] <= 15 else 0

df["GOT"] = df.apply(create_got_label, axis=1)

In [ ]:
df = pd.get_dummies(df, columns=["Cluster", "Type"], drop_first=True)

In [ ]:
feature_columns = [
    "proposal_delay_months",
    "research_delay_months",
    "submission_delay_months",
    "total_duration_semesters"
] + [col for col in df.columns if "Cluster_" in col or "Type_" in col]

X = df[feature_columns]
y = df["GOT"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

In [ ]:
joblib.dump(model, "dev.ipynb")